In [ ]:
import sys
import os
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('..'))

from src.config import ExperimentConfig, Hyperparameters
from src.trainer import Trainer
from src.inference import InferenceEngine

hyperparams = Hyperparameters(
    num_iterations=20,
    group_size=4,
    vllm_batch_size=2,
    learning_rate=5e-6,
    max_tokens=256,
    lora_rank=16,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    rl_coef=1.0,
    clip_ratio=0.2,
    temperature=0.8,
    top_p=0.9,
    gpu_memory_utilization=0.2
)

config = ExperimentConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    judge_model="Qwen/Qwen2.5-Math-1.5B-Instruct",
    env="gsm8k",
    algo="grpo",
    log_dir="./grpo_alignment",
    hyperparameters=hyperparams
)

In [ ]:
trainer = Trainer(config)
losses = trainer.train()
del trainer # Free up the memory buffers for inference!

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o', linestyle='-', color='b')
plt.title('GRPO Training Loss Curve')
plt.xlabel('Iteration Step')
plt.ylabel('Clipped Surrogate Loss')
plt.grid(True)
plt.show()

In [ ]:
engine_base = InferenceEngine(config)
base_reward = engine_base.evaluate_env(num_samples=10)
del engine_base

In [ ]:
engine_tuned = InferenceEngine(config, adapter_path="./grpo_alignment")
tuned_reward = engine_tuned.evaluate_env(num_samples=10)

In [ ]:
models = ['Base Qwen-0.5B', 'GRPO Aligned Qwen']
rewards = [base_reward, tuned_reward]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, rewards, color=['gray', 'green'])
plt.title('Environment Pass Rate (Test Split Evaluation)')
plt.ylabel('Average Reward (Accuracy)')
plt.ylim(0, 1.05)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval, f"{yval*100:.1f}%", va='bottom', ha='center')

plt.show()

In [ ]:
prompt = "Let's think step by step. A store has 15 apples and sells 7. Then they receive a shipment of 20 apples. How many apples do they have now?"
engine_tuned.chat(prompt)